# Negative control analysis

We take the generated negative controls and summarize the results.

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns

functional_or_path = 'results/functional-NC-OR-estimates_2026-10-08.parquet'
excalibr_or_path = 'results/excalibr-NC-OR-estimates_2026-10-08.parquet'
vep_or_path = 'results/vep-NC-OR-estimates_2026-10-08.parquet'
tp_or_path = 'results/tp-NC-OR-estimates_2026-10-08.parquet'

## Are any ORs significant?
We look at the p-value distribution of the combined table

In [3]:
nc_or_df = pd.concat([
    pd.read_parquet(functional_or_path),
    pd.read_parquet(excalibr_or_path),
    pd.read_parquet(vep_or_path),
    pd.read_parquet(tp_or_path)
])

In [4]:
nc_or_df

In [5]:
nc_or_df.describe()

In [6]:
nc_or_df[nc_or_df['p-value'] < 0.05]

In [7]:
nc_or_df['- Log_10 p-value'] = -np.log10(nc_or_df['p-value'])

In [8]:
carrier_min = nc_or_df[['Cases with variants', 'Controls with variants']].min(axis='columns')

In [9]:
nc_or_df['min(carrier cases/controls, carrier controls)'] = '>20'
nc_or_df.loc[carrier_min <= 20, 'min(carrier cases, carrier controls)'] = '11-20'
nc_or_df.loc[carrier_min <= 10, 'min(carrier cases, carrier controls)'] = '6-10'
nc_or_df.loc[carrier_min <= 5, 'min(carrier cases, carrier controls)'] = '3-5'
nc_or_df.loc[carrier_min == 2, 'min(carrier cases, carrier controls)'] = '2'
nc_or_df.loc[carrier_min == 1, 'min(carrier cases, carrier controls)'] = '1'
nc_or_df.loc[carrier_min == 0, 'min(carrier cases, carrier controls)'] = '0'

nc_or_df['min(carrier cases, carrier controls)'] = pd.Categorical(
    nc_or_df['min(carrier cases, carrier controls)'],
    ['0',  '1', '2', '3-5', '6-10', '11-20', '>20'],
    ordered=True
)

In [30]:
ax = sns.relplot(
    nc_or_df,
    x='LogOR',
    y='- Log_10 p-value',
    hue='min(carrier cases, carrier controls)',
    facet_kws=dict(legend_out=False)
)
sns.move_legend(ax, 'right', ncol=2, bbox_to_anchor=(1,0.7))

In [49]:
sns.relplot(
    nc_or_df[carrier_min > 1],
    x='LogOR',
    y='- Log_10 p-value',
    hue='min(carrier cases, carrier controls)'
)

In [45]:
sns.displot(
    nc_or_df[carrier_min > 1],
    x='p-value',
    #hue='min(carrier cases, carrier controls)',
    kind='ecdf'
)

In [23]:
sns.relplot(
    nc_or_df,
    x='LogOR',
    y='- Log_10 p-value',
    hue='>20 carrier cases and carrier controls'
)

In [19]:
carrier_min = nc_or_df[['Cases with variants', 'Controls with variants']].min(axis='columns')
nc_or_df['ccounts'] = '>20 carrier cases and carrier controls'
nc_or_df.loc[carrier_min <= 20 ,'ccounts'] = '<=20 carrier cases or carrier contols'
nc_or_df.loc[carrier_min == 0, 'ccounts'] = '0 carrier cases or carrier controls'

In [20]:
sns.relplot(
    nc_or_df,
    x='LogOR',
    y='- Log_10 p-value',
    hue='ccounts'
)

In [9]:
nc_or_df[nc_or_df['p-value'] < 0.05].index.to_frame()['Mismatched Phenotype'].value_counts()

In [10]:
import seaborn as sns

In [25]:
sns.relplot(
    nc_or_df,
    x='p-value',
    y='Cases with variants'
)

In [14]:
sns.relplot(
    nc_or_df,
    x='p-value',
    y='Cases with variants',
    row='Group/Range'
)